In [ ]:
# SETTINGS — edit this cell in your local working copy.
from pathlib import Path
import os

MIMIC_DIR = Path(os.environ.get("MIMIC_DIR") or Path.home() / "mimic-iv-3.1")  # Contains hosp/ and icu/
OUTPUT_DIR = Path(os.environ["LOCALAPPDATA"]) / "AKI-feature-extraction" / "output"
MODE = "pilot"               # "pilot" or "full"
PILOT_N = 200
PILOT_SEED = 42
USE_SYNTHETIC = False        # True uses small generated CSVs and runs assertions
SHOW_STAY_TRACE = False     # True displays individual records. Enable only in the local copy.
TRACE_STAY_ID = None         # None selects a stay with recorded heart rate
LOOKBACK_HOURS = 6          # Positive integer, separate from the fixed six-hour stride
MIN_AGE = 18
THREADS = 4
MEMORY_LIMIT = "2GB"

# MIMIC-IV v3.1 ICU feature extraction

Run the cells top to bottom. Use Python 3.11+ with DuckDB 1.4.4 and ipykernel. Keep this clean copy output-free. Run a separate copy outside OneDrive. The notebook writes the database, Parquet files and spill files under the resolved output directory. Keep those files and the executed working notebook local.

Review 2 (FR-02, FR-05 and FR-06) specifies adult stays, hourly alignment and a six-hour stride. The Review 2 PDF and current request use a six-hour lookback. The alternate Review 2 PPTX uses a 24-hour lookback. Set LOOKBACK_HOURS = 24 to evaluate that alternative at the same cutoffs. This version includes two renal slopes and four documented ICU antibiotic start counts.

In [ ]:
import csv
import hashlib
import json
from time import perf_counter

import duckdb

assert MODE in {"pilot", "full"} and isinstance(PILOT_N, int) and PILOT_N > 0
assert type(LOOKBACK_HOURS) is int and LOOKBACK_HOURS > 0
assert isinstance(PILOT_SEED, int) and isinstance(MIN_AGE, int)
assert isinstance(THREADS, int) and THREADS > 0
output_root = Path(OUTPUT_DIR).expanduser().resolve()
mimic_root = Path(MIMIC_DIR).expanduser().resolve()
sync_roots = [Path(path).resolve() for name in ("OneDrive", "OneDriveCommercial", "OneDriveConsumer")
              if (path := os.environ.get(name))]
if (any(part.lower().startswith("onedrive") for part in output_root.parts)
        or any(output_root.is_relative_to(root) for root in sync_roots)
        or output_root.is_relative_to(mimic_root)):
    raise ValueError("Choose a local OUTPUT_DIR outside OneDrive and the MIMIC source folder.")

run_dir = output_root / ("synthetic" if USE_SYNTHETIC else MODE)
run_dir.mkdir(parents=True, exist_ok=True)
spill_dir = run_dir / "tmp"
spill_dir.mkdir(exist_ok=True)

def literal(value):
    """Quote a filesystem path or string for SQL."""
    return "'" + str(value).replace("'", "''") + "'"

def run_step(label, sql):
    print(f"{label} …", flush=True)
    started = perf_counter()
    result = con.execute(sql)
    print(f"  {perf_counter() - started:.2f} s", flush=True)
    return result

def parquet(path):
    return f"read_parquet({literal(path)})"

def write_parquet(query, path, label):
    staging = path.with_suffix(".partial.parquet")
    try:
        run_step(label, f"COPY ({query}) TO {literal(staging)} (FORMAT PARQUET, COMPRESSION ZSTD)")
        staging.replace(path)
    finally:
        staging.unlink(missing_ok=True)

if "con" in globals():
    con.close()
con = duckdb.connect(str(run_dir / "cache.duckdb"))
con.execute(f"SET temp_directory = {literal(spill_dir)}")
con.execute(f"SET threads = {THREADS}")
con.execute(f"SET memory_limit = {literal(MEMORY_LIMIT)}")
con.execute("SET preserve_insertion_order = false")
con.execute("PRAGMA disable_progress_bar")
print(f"DuckDB {duckdb.__version__} · {run_dir}")

## Source files and feature mapping

Point MIMIC_DIR at the extracted release root containing **hosp** and **icu**. For each table, prefer CSV.GZ when both formats exist. Set USE_SYNTHETIC to True to exercise the same CSV-to-Parquet path using fictional events.

| Feature | Source and items | Unit or definition |
|---|---|---|
| Age at hospital admission | patients.anchor_age, patients.anchor_year and admissions.admittime | Approximate years, computed as anchor age plus admission year minus anchor year. MIMIC groups patients over 89 at anchor age 91. |
| Recorded gender | patients.gender | Source category, exposed as recorded_gender. |
| Heart rate | chartevents 220045 | bpm |
| Systolic / diastolic BP | chartevents arterial 220050 / 220051 and noninvasive 220179 / 220180 | mmHg, with both methods eligible |
| Serum creatinine | labevents 50912, Creatinine in Blood | mg/dL |
| Antibiotic administration segment starts | inputevents 225798 vancomycin, 225840 amikacin, 225875 gentamicin, 225902 tobramycin | Distinct documented order segments per drug |
| Urine output | outputevents 226557, 226558, 226559, 226560, 226561, 226563, 226564, 226565, 226567, 226584 | mL from the listed drainage or void measurements. Irrigant input and mixed irrigant/urine output are excluded. |

The notebook checks the item mapping against the installed dictionaries. It accepts finite positive vitals and creatinine and finite nonnegative urine volumes in the expected units. The cache keeps the source unit and raw numeric value even when an observation is excluded from aggregation. See the [MIMIC-IV v3.1 release](https://physionet.org/content/mimiciv/3.1/) and [MIT-LCP vital sign mapping](https://github.com/MIT-LCP/mimic-code/blob/main/mimic-iv/concepts/measurement/vitalsign.sql).

In [ ]:
MODULES = {
    "icustays": "icu", "chartevents": "icu", "outputevents": "icu", "d_items": "icu",
    "inputevents": "icu", "patients": "hosp", "admissions": "hosp", "labevents": "hosp", "d_labitems": "hosp",
}
# feature, source table, dictionary label, canonical unit
ITEMS = {
    220045: ("heart_rate", "chartevents", "Heart Rate", "bpm"),
    220050: ("sbp", "chartevents", "Arterial Blood Pressure systolic", "mmHg"),
    220051: ("dbp", "chartevents", "Arterial Blood Pressure diastolic", "mmHg"),
    220179: ("sbp", "chartevents", "Non Invasive Blood Pressure systolic", "mmHg"),
    220180: ("dbp", "chartevents", "Non Invasive Blood Pressure diastolic", "mmHg"),
    50912: ("creatinine", "labevents", "Creatinine", "mg/dL"),
}
DRUGS = {
    225798: ("vancomycin", "Vancomycin"),
    225840: ("amikacin", "Amikacin"),
    225875: ("gentamicin", "Gentamicin"),
    225902: ("tobramycin", "Tobramycin"),
}
URINE_LABELS = {
    226557: "R Ureteral Stent", 226558: "L Ureteral Stent", 226559: "Foley",
    226560: "Void", 226561: "Condom Cath", 226563: "Suprapubic", 226564: "R Nephrostomy",
    226565: "L Nephrostomy", 226567: "Straight Cath", 226584: "Ileoconduit",
}
ITEMS.update({item: ("urine", "outputevents", label, "mL") for item, label in URINE_LABELS.items()})

def build_fixture(root):
    def write(table, columns, rows):
        folder = root / MODULES[table]
        folder.mkdir(parents=True, exist_ok=True)
        with (folder / f"{table}.csv").open("w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(columns.split())
            writer.writerows(rows)

    def stamp(clock):
        return f"2150-01-01 {clock}:00" if clock else None

    write("icustays", "subject_id hadm_id stay_id intime outtime", [
        (1, 10, 100, stamp("01:15"), stamp("19:15")),
        (1, 10, 101, stamp("13:00"), stamp("22:00")),
    ])
    write("patients", "subject_id gender anchor_age anchor_year", [(1, "F", 45, 2149)])
    write("admissions", "subject_id hadm_id admittime", [(1, 10, stamp("00:00"))])
    write("d_items", "itemid label linksto unitname", [
        (item, label, source, unit) for item, (_, source, label, unit) in ITEMS.items()
        if source != "labevents"
    ] + [(item, label, "inputevents", "dose") for item, (_, label) in DRUGS.items()])
    write("d_labitems", "itemid label fluid", [(50912, "Creatinine", "Blood")])
    write("chartevents", "subject_id hadm_id stay_id itemid charttime storetime valuenum valueuom", [
        (1, 10, 100, item, stamp(chart), stamp(store), value, ITEMS[item][3])
        for item, chart, store, value in [
            (220045, "01:15", "01:16", 70),
            (220045, "07:14", "07:15", 80),
            (220045, "07:14", "07:16", 99),  # Too late for the first window.
            (220045, "07:14", None, 98),     # Availability unknown.
            (220045, "07:15", "07:15", 90),  # Belongs to the next window.
            (220045, "07:16", "07:16", 92),
            (220179, "05:00", "05:00", 120),
            (220180, "05:00", "05:00", 70),
        ]
    ])
    write("outputevents", "subject_id hadm_id stay_id itemid charttime storetime value valueuom", [
        (1, 10, 100, 226559, stamp("03:00"), stamp("03:01"), 0, "mL"),
        (1, 10, 100, 226559, stamp("05:00"), stamp("05:01"), 40, "mL"),
        (1, 10, 100, 226559, stamp("05:05"), stamp("05:06"), 20, "mL"),
        (1, 10, 101, 226560, stamp("15:00"), stamp("15:01"), 25, "mL"),
        (1, 10, 101, 226560, stamp("16:00"), stamp("16:01"), 30, "mL"),
        (1, 10, 101, 226560, stamp("17:00"), stamp("17:01"), 100, "L"),
    ])
    write("inputevents", "subject_id hadm_id stay_id itemid starttime endtime storetime amount amountuom orderid statusdescription", [
        (1, 10, 100, 225798, stamp("04:00"), stamp("10:00"), stamp("04:05"), 1, "dose", 1, "FinishedRunning"),
        (1, 10, 100, 225798, stamp("04:00"), stamp("10:00"), stamp("04:05"), 1, "dose", 1, "FinishedRunning"),
        (1, 10, 100, 225875, stamp("06:00"), stamp("06:30"), stamp("08:00"), 1, "dose", 2, "FinishedRunning"),
        (1, 10, 100, 225840, stamp("06:00"), stamp("06:30"), stamp("06:05"), 1, "dose", 3, "Rewritten"),
        (1, 10, 100, 225902, stamp("07:15"), stamp("07:45"), stamp("07:16"), 1, "dose", 4, "FinishedRunning"),
    ])
    write("labevents", "labevent_id subject_id hadm_id itemid charttime storetime valuenum valueuom", [
        (idx, 1, 10, 50912, stamp(chart), stamp(store), value, "mg/dL")
        for idx, (chart, store, value) in enumerate([
            ("00:45", "01:00", 0.6),  # Pre-ICU, excluded.
            ("03:00", "04:00", 1.0),
            ("05:00", "05:30", 1.4),
            ("07:00", "09:00", 1.9),  # Late result, excluded from its measurement window.
            ("13:05", "13:10", 2.0),  # Falls in both stays, assigned to the later admission.
        ], start=1)
    ])

source_root = mimic_root
if USE_SYNTHETIC:
    source_root = run_dir / "fixture"
    build_fixture(source_root)

paths = {}
for table, module in MODULES.items():
    candidates = [source_root / module / f"{table}{suffix}" for suffix in (".csv.gz", ".csv")]
    path = next((candidate for candidate in candidates if candidate.is_file()), None)
    if path is None:
        raise FileNotFoundError(f"Missing {table}. Set MIMIC_DIR to the folder containing hosp and icu. Looked in {candidates}")
    paths[table] = path

def source(table):
    # Projection/filtering happens in DuckDB. Gzip still needs a sequential source scan.
    return f"read_csv({literal(paths[table])}, header=true, delim=',', all_varchar=true, sample_size=2048)"

dictionary = {
    int(item): (label, link, unit)
    for item, label, link, unit in con.execute(f"""
        SELECT itemid, label, linksto, unitname FROM {source('d_items')}
        UNION ALL
        SELECT itemid, label, 'labevents', fluid FROM {source('d_labitems')}
    """).fetchall()
}
for item, (_, label) in DRUGS.items():
    assert dictionary.get(item) == (label, "inputevents", "dose"), (item, dictionary.get(item))
for item, (_, table, label, unit) in ITEMS.items():
    expected = (label, table, "Blood" if table == "labevents" else unit)
    assert dictionary.get(item) == expected, (item, dictionary.get(item), expected)
print(f"Verified {len(ITEMS) + len(DRUGS)} item mappings. Creatinine units are checked on event rows.")

## ICU stay cohort

Use one row per ICU stay. The provisional rule includes hospital admissions with estimated age at least 18 and ICU stays with room for a first prediction strictly before discharge. Repeat ICU stays can qualify. The pilot orders by a seeded MD5 of stay ID and takes the first PILOT_N stays. It does not use AKI status.

Keep all valid ICU intervals available for lab assignment, including intervals outside the sampled cohort. This keeps assignment consistent across pilot and full runs.

In [ ]:
spine_sql = f"""
CREATE OR REPLACE TEMP TABLE all_stays AS
SELECT subject_id::BIGINT AS subject_id, hadm_id::BIGINT AS hadm_id,
       stay_id::BIGINT AS stay_id, intime::TIMESTAMP AS intime, outtime::TIMESTAMP AS outtime
FROM {source('icustays')}
WHERE stay_id IS NOT NULL AND subject_id IS NOT NULL AND hadm_id IS NOT NULL
  AND intime::TIMESTAMP < outtime::TIMESTAMP
"""
run_step("Load ICU stay spine", spine_sql)
assert con.execute("SELECT count(*) = count(DISTINCT stay_id) FROM all_stays").fetchone()[0]

limit = f"ORDER BY md5('{PILOT_SEED}:' || stay_id::VARCHAR), stay_id LIMIT {PILOT_N}" if MODE == "pilot" else ""
cohort_sql = f"""
CREATE OR REPLACE TEMP TABLE cohort AS
WITH demographics AS (
    SELECT s.*, p.gender AS recorded_gender,
           p.anchor_age::INTEGER + year(a.admittime::TIMESTAMP) - p.anchor_year::INTEGER
               AS age_at_admission
    FROM all_stays s
    JOIN {source('admissions')} a ON s.subject_id = a.subject_id::BIGINT AND s.hadm_id = a.hadm_id::BIGINT
    JOIN {source('patients')} p ON s.subject_id = p.subject_id::BIGINT
)
SELECT * FROM demographics
WHERE age_at_admission >= {MIN_AGE} AND intime + INTERVAL '6 hours' < outtime
{limit}
"""
run_step("Build adult ICU cohort", cohort_sql)
cohort_size, unique_stays = con.execute("SELECT count(*), count(DISTINCT stay_id) FROM cohort").fetchone()
assert cohort_size == unique_stays and cohort_size > 0, "Cohort must contain unique stays."
print(f"{cohort_size:,} selected ICU stays")

## Select observations and cache Parquet

The heart-rate scan is the first event step. Blood pressure uses the same chart scan. ICU chart and output rows join on the full stay key and must fall inside the ICU interval. For labs, match subject and hospital admission, then require charttime in [intime, outtime). If ICU intervals overlap, choose the latest intime, then lowest stay_id. Apply the pilot filter after assigning a lab. Labs without hadm_id and pre-ICU measurements are excluded.

Availability is the later of charttime and storetime when both exist. Rows with missing storetime stay visible in the cache and are ineligible for features. Treat storetime as the release proxy available in the source. Each run prints source file size and scan/write time on cache misses. Even a 200-stay pilot traverses each event CSV on the first scan. Cache hits skip these event scans. File paths, sizes, modification times, cohort configuration and scan SQL determine the cache key. Use one kernel at a time per output directory.

In [ ]:
con.execute("""
CREATE OR REPLACE TEMP TABLE item_map (
    itemid INTEGER, feature VARCHAR, source_table VARCHAR, expected_unit VARCHAR
)
""")
con.executemany("INSERT INTO item_map VALUES (?, ?, ?, ?)", [
    (item, feature, table, unit) for item, (feature, table, _, unit) in ITEMS.items()
])

def observation_query(table):
    lab = table == "labevents"
    items = ", ".join(str(item) for item, (_, src, _, _) in ITEMS.items() if src == table)
    value_col = "value" if table == "outputevents" else "valuenum"
    source_id = "labevent_id::BIGINT" if lab else "NULL::BIGINT"
    stay_col = "" if lab else ", stay_id::BIGINT AS stay_id"
    if lab:
        assignment = """
        SELECT e.*, s.stay_id
        FROM events e
        SEMI JOIN cohort c USING (subject_id, hadm_id)
        JOIN all_stays s ON e.subject_id = s.subject_id AND e.hadm_id = s.hadm_id
                       AND e.charttime >= s.intime AND e.charttime < s.outtime
        QUALIFY row_number() OVER (
            PARTITION BY source_event_id ORDER BY s.intime DESC, s.stay_id
        ) = 1
        """
    else:
        assignment = """
        SELECT e.* FROM events e
        JOIN cohort c USING (subject_id, hadm_id, stay_id)
        WHERE e.charttime >= c.intime AND e.charttime < c.outtime
        """
    return f"""
        WITH events AS (
            SELECT subject_id::BIGINT AS subject_id, hadm_id::BIGINT AS hadm_id {stay_col},
                   {source_id} AS source_event_id, itemid::INTEGER AS itemid,
                   charttime::TIMESTAMP AS charttime, storetime::TIMESTAMP AS storetime,
                   try_cast({value_col} AS DOUBLE) AS raw_value, valueuom
            FROM {source(table)}
            WHERE itemid::INTEGER IN ({items})
        ), assigned AS ({assignment})
        SELECT e.subject_id, e.hadm_id, e.stay_id, m.source_table, e.source_event_id,
               e.itemid, m.feature, e.charttime, e.storetime,
               CASE WHEN e.storetime IS NOT NULL THEN greatest(e.charttime, e.storetime) END AS available_time,
               e.raw_value, e.valueuom, m.expected_unit,
               CASE WHEN lower(trim(e.valueuom)) = lower(m.expected_unit)
                         AND isfinite(e.raw_value)
                         AND (e.raw_value > 0 OR (m.feature = 'urine' AND e.raw_value = 0))
                    THEN e.raw_value END AS feature_value
        FROM assigned e
        SEMI JOIN cohort c USING (stay_id)
        JOIN item_map m USING (itemid)
    """

def cache_observations(table, query=None):
    if query is None:
        query = observation_query(table)
    inputs = [table, "icustays", "patients", "admissions"]
    fingerprint = [
        (str(paths[name]), paths[name].stat().st_size, paths[name].stat().st_mtime_ns)
        for name in inputs
    ]
    key = hashlib.sha256(json.dumps([
        fingerprint, MODE, spine_sql, cohort_sql, sorted(ITEMS.items()), query, duckdb.__version__,
    ]).encode()).hexdigest()[:20]
    path = run_dir / f"{table}-{key}.parquet"
    if path.exists():
        print(f"Cache hit: {path.name} (source scan skipped)")
    else:
        size_mib = paths[table].stat().st_size / (1024 ** 2)
        write_parquet(query, path, f"Scan {paths[table].name} ({size_mib:,.1f} MiB) and write selected rows")
    return path

chart_cache = cache_observations("chartevents")
print("Heart rate observations with valid numeric values:",
      con.execute(f"SELECT count(feature_value) FROM {parquet(chart_cache)} WHERE feature = 'heart_rate'").fetchone()[0])

In [ ]:
output_cache = cache_observations("outputevents")
lab_cache = cache_observations("labevents")
con.execute(f"""
CREATE OR REPLACE TEMP VIEW observations AS
SELECT * FROM {parquet(chart_cache)}
UNION ALL SELECT * FROM {parquet(output_cache)}
UNION ALL SELECT * FROM {parquet(lab_cache)}
""")
print("Cached observations by feature and source unit:")
for row in con.execute("""
    SELECT feature, valueuom, count(*) AS cached_rows,
           count(feature_value) AS valid_values,
           count(*) FILTER (WHERE available_time IS NULL) AS unknown_availability
    FROM observations GROUP BY ALL ORDER BY feature, cached_rows DESC
""").fetchall():
    print(row)

## Documented antibiotic starts

Map four ICU inputevents items using the local dictionary. Count distinct orderid values per drug whose segment starts in the lookback and whose starttime and storetime are both available at the cutoff. Require the ICU stay key, a positive finite source amount, and a known orderid. Exclude Rewritten and Flushed rows. These retrospective record-quality fields are used to identify invalid segments. Their own change history is unavailable. The feature uses no end time or completed dose. A zero means no qualifying segment was recorded by cutoff. Ongoing segments started before the lookback, eMAR-only administrations, contrast, and prescriptions are outside this mapping. See [MIMIC inputevents](https://mimic.mit.edu/docs/iv/modules/icu/inputevents.html).

In [ ]:
drug_ids = ", ".join(map(str, DRUGS))
drug_query = f"""
SELECT c.subject_id, c.hadm_id, c.stay_id, e.itemid::INTEGER AS itemid,
       e.orderid::BIGINT AS orderid, e.starttime::TIMESTAMP AS charttime,
       e.storetime::TIMESTAMP AS storetime, e.endtime::TIMESTAMP AS endtime,
       CASE WHEN e.storetime IS NOT NULL
            THEN greatest(e.starttime::TIMESTAMP, e.storetime::TIMESTAMP) END AS available_time,
       try_cast(e.amount AS DOUBLE) AS amount, e.amountuom, e.statusdescription
FROM {source('inputevents')} e
JOIN cohort c ON e.subject_id::BIGINT = c.subject_id
             AND e.hadm_id::BIGINT = c.hadm_id AND e.stay_id::BIGINT = c.stay_id
WHERE e.itemid::INTEGER IN ({drug_ids})
  AND e.starttime::TIMESTAMP >= c.intime AND e.starttime::TIMESTAMP < c.outtime
"""
drug_cache = cache_observations("inputevents", drug_query)
con.execute(f"CREATE OR REPLACE TEMP VIEW drug_observations AS SELECT * FROM {parquet(drug_cache)}")
print("Cached drug segments by item, source unit and status:")
print(con.execute("""
    SELECT itemid, amountuom, statusdescription, count(*) FROM drug_observations
    GROUP BY ALL ORDER BY itemid, count(*) DESC
""").fetchall())

## Build one row per prediction cutoff

ICU-relative hour k contains measurements from intime + k hours (inclusive) to intime + (k+1) hours (exclusive). The default windows use hours 0–5, then 6–11, and so on. LOOKBACK_HOURS can be changed (e.g. 24 to match the Review 2 PPTX). Earlier cutoffs then use partial ICU history. The stride stays at six hours. Each lookback is [max(intime, cutoff − lookback), cutoff). A measurement exactly at a cutoff is excluded from that cutoff. A cutoff is emitted every six hours starting at intime + 6 hours, strictly before outtime.

Use observations from the included hourly bins whose available_time is at or before the cutoff. A late result can contribute to a later cutoff if its measurement still falls inside that cutoff's lookback. Take the latest eligible heart rate, systolic BP, diastolic BP and creatinine by charttime. Break ties by later availability, then lower item ID, then higher value. Sum eligible recorded urine volumes across the six bins. Each count is the number of eligible observations. A window with no eligible records for a feature has a null value and zero count. A recorded urine volume of zero contributes a count and a value of zero.

Calculate least-squares creatinine slope on one value per distinct measurement timestamp (latest availability, then higher value). Calculate least-squares urine slope across observed hourly volume totals. Leave unobserved hours out of the fit and return null for either slope when fewer than two distinct time points are available. Urine slope is the change in hourly recorded mL per hour of ICU time. No weight normalization is applied.

In [ ]:
run_step("Generate six-hour cutoffs", f"""
CREATE OR REPLACE TEMP TABLE windows AS
SELECT c.*, n AS window_index,
       greatest(c.intime, c.intime + ((n + 1) * 6 - {LOOKBACK_HOURS}) * INTERVAL '1 hour') AS window_start,
       c.intime + (n + 1) * INTERVAL '6 hours' AS prediction_time
FROM cohort c
CROSS JOIN LATERAL range(0, date_diff('microsecond', c.intime, c.outtime) // 21600000000) AS r(n)
WHERE c.intime + (n + 1) * INTERVAL '6 hours' < c.outtime
""")
con.execute("""
CREATE OR REPLACE TEMP VIEW hourly_observations AS
SELECT o.*, date_diff('microsecond', c.intime, o.charttime) // 3600000000 AS hour_index
FROM observations o JOIN cohort c USING (stay_id)
""")
con.execute("""
CREATE OR REPLACE TEMP VIEW eligible_observations AS
SELECT o.*, w.prediction_time
FROM hourly_observations o
JOIN windows w ON o.stay_id = w.stay_id AND o.hour_index < (w.window_index + 1) * 6
              AND o.charttime >= w.window_start
WHERE o.available_time <= w.prediction_time AND o.feature_value IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW eligible_drugs AS
SELECT d.*, w.prediction_time
FROM drug_observations d JOIN windows w USING (stay_id)
WHERE d.charttime >= w.window_start AND d.charttime < w.prediction_time
  AND d.available_time <= w.prediction_time
  AND d.orderid IS NOT NULL AND isfinite(d.amount) AND d.amount > 0
  AND lower(trim(d.statusdescription)) NOT IN ('rewritten', 'flushed')
""")
FEATURE_COLUMNS = {
    "heart_rate": "heart_rate_bpm", "sbp": "sbp_mmhg", "dbp": "dbp_mmhg",
    "creatinine": "creatinine_mg_dl", "urine": "urine_output_ml",
}
aggregates = []
for feature, column in FEATURE_COLUMNS.items():
    aggregate = "sum(feature_value)" if feature == "urine" else (
        "first(feature_value ORDER BY charttime DESC, available_time DESC, itemid, feature_value DESC)"
    )
    condition = f"FILTER (WHERE feature = '{feature}')"
    aggregates.extend([f"{aggregate} {condition} AS {column}",
                       f"count(*) {condition} AS {feature}_n"])
aggregate_sql = ",\n           ".join(aggregates)
projection = ", ".join(f"a.{column}, coalesce(a.{feature}_n, 0) AS {feature}_n"
                       for feature, column in FEATURE_COLUMNS.items())
drug_aggregates = ", ".join(
    f"count(DISTINCT orderid) FILTER (WHERE itemid = {item}) AS {drug}_segments_n"
    for item, (drug, _) in DRUGS.items()
)
drug_projection = ", ".join(
    f"coalesce(d.{drug}_segments_n, 0) AS {drug}_segments_n" for drug, _ in DRUGS.values()
)
SLOPE_COLUMNS = ["creatinine_slope_mg_dl_per_hour", "urine_hourly_ml_slope"]
DRUG_COLUMNS = [f"{drug}_segments_n" for drug, _ in DRUGS.values()]
feature_query = f"""
WITH aggregated AS (
    SELECT stay_id, prediction_time, {aggregate_sql}
    FROM eligible_observations GROUP BY stay_id, prediction_time
), creatinine_times AS (
    SELECT stay_id, prediction_time, charttime,
           first(feature_value ORDER BY available_time DESC, feature_value DESC) AS value
    FROM eligible_observations WHERE feature = 'creatinine' GROUP BY ALL
), creatinine_trend AS (
    SELECT stay_id, prediction_time, regr_slope(
        value, date_diff('microsecond', prediction_time, charttime) / 3600000000.0
    ) AS creatinine_slope_mg_dl_per_hour
    FROM creatinine_times GROUP BY stay_id, prediction_time HAVING count(*) >= 2
), urine_hours AS (
    SELECT stay_id, prediction_time, hour_index, sum(feature_value) AS volume
    FROM eligible_observations WHERE feature = 'urine' GROUP BY ALL
), urine_trend AS (
    SELECT stay_id, prediction_time, regr_slope(volume, hour_index) AS urine_hourly_ml_slope
    FROM urine_hours GROUP BY stay_id, prediction_time HAVING count(*) >= 2
), drugs AS (
    SELECT stay_id, prediction_time, {drug_aggregates}
    FROM eligible_drugs GROUP BY stay_id, prediction_time
)
SELECT w.subject_id, w.hadm_id, w.stay_id, w.intime, w.outtime, w.window_start,
       w.prediction_time, w.age_at_admission, w.recorded_gender, {projection},
       ct.creatinine_slope_mg_dl_per_hour, ut.urine_hourly_ml_slope, {drug_projection}
FROM windows w LEFT JOIN aggregated a USING (stay_id, prediction_time)
LEFT JOIN creatinine_trend ct USING (stay_id, prediction_time)
LEFT JOIN urine_trend ut USING (stay_id, prediction_time)
LEFT JOIN drugs d USING (stay_id, prediction_time)
"""
features_path = run_dir / "features.parquet"
write_parquet(feature_query, features_path, "Aggregate features and write features.parquet")
con.execute(f"CREATE OR REPLACE TEMP VIEW features AS SELECT * FROM {parquet(features_path)}")

## Validate and inspect the output

The checks compare the feature keys with the expected stay grid, verify lab uniqueness and audit the temporal eligibility conditions. The fixture additionally checks exact expected values, observations at the boundary, late and unknown availability, overlapping ICU stays, and recorded zero versus missing urine. Enable SHOW_STAY_TRACE in the settings cell only in your local copy. The trace prints a feature row beside selected source measurements and their inclusion decisions.

In [ ]:
row_count, stay_count = con.execute("SELECT count(*), count(DISTINCT stay_id) FROM features").fetchone()
assert (row_count, stay_count) == (con.execute("SELECT count(*) FROM windows").fetchone()[0], cohort_size)
assert con.execute("""
    SELECT count(*) FROM (
        SELECT stay_id, prediction_time FROM features GROUP BY ALL HAVING count(*) > 1
    )
""").fetchone()[0] == 0
assert con.execute("""
    SELECT count(*) FROM windows ANTI JOIN features USING (stay_id, prediction_time)
""").fetchone()[0] == 0
assert con.execute(f"""
    SELECT count(*) FROM features
    WHERE prediction_time >= outtime OR window_start < intime
       OR window_start <> greatest(intime, prediction_time - {LOOKBACK_HOURS} * INTERVAL '1 hour')
       OR date_diff('microsecond', intime, prediction_time) % 21600000000 <> 0
""").fetchone()[0] == 0
assert con.execute("""
    SELECT count(*) FROM eligible_observations o JOIN windows w USING (stay_id, prediction_time)
    WHERE o.charttime < w.window_start OR o.charttime >= w.prediction_time
       OR o.available_time IS NULL OR o.available_time > w.prediction_time
""").fetchone()[0] == 0
assert con.execute("""
    SELECT count(*) = count(DISTINCT source_event_id)
    FROM observations WHERE source_table = 'labevents'
""").fetchone()[0]
print(f"{row_count:,} feature rows across {stay_count:,} stays. Duplicate, boundary and availability checks passed.")
assert con.execute("""
    SELECT count(*) FROM eligible_drugs d JOIN windows w USING (stay_id, prediction_time)
    WHERE d.charttime < w.window_start OR d.charttime >= w.prediction_time
       OR d.available_time IS NULL OR d.available_time > w.prediction_time
""").fetchone()[0] == 0
print("Missing rows by feature:")
for column in [*FEATURE_COLUMNS.values(), *SLOPE_COLUMNS, "age_at_admission", "recorded_gender"]:
    missing = con.execute(f"SELECT count(*) - count({column}) FROM features").fetchone()[0]
    print(f"  {column:20} {missing:8,} ({missing / row_count:6.1%})")
print(f"Features: {features_path}")

for column in DRUG_COLUMNS:
    positive = con.execute(f"SELECT count(*) FROM features WHERE {column} > 0").fetchone()[0]
    print(f"  {column:24} positive in {positive:,} windows ({positive / row_count:.2%})")
if USE_SYNTHETIC and LOOKBACK_HOURS == 6 and (MODE == "full" or PILOT_N >= 2) and MIN_AGE <= 46:
    actual = con.execute("""
        SELECT stay_id, heart_rate_bpm, heart_rate_n, sbp_mmhg, dbp_mmhg,
               creatinine_mg_dl, creatinine_n, urine_output_ml, urine_n,
               age_at_admission, recorded_gender
        FROM features ORDER BY stay_id, prediction_time
    """).fetchall()
    assert actual == [
        (100, 80.0, 2, 120.0, 70.0, 1.4, 2, 60.0, 3, 46, "F"),
        (100, 92.0, 2, None, None, None, 0, None, 0, 46, "F"),
        (101, None, 0, None, None, 2.0, 1, 55.0, 2, 46, "F"),
    ], actual
    assert con.execute("""
        SELECT source_event_id, stay_id FROM observations
        WHERE source_table = 'labevents' ORDER BY source_event_id
    """).fetchall() == [(2, 100), (3, 100), (4, 100), (5, 101)]
    assert con.execute("""
        SELECT raw_value, hour_index FROM hourly_observations
        WHERE stay_id = 100 AND raw_value IN (80, 90, 92) ORDER BY raw_value
    """).fetchall() == [(80.0, 5), (90.0, 6), (92.0, 6)]
    from math import isclose
    renal = con.execute("""
        SELECT creatinine_slope_mg_dl_per_hour, urine_hourly_ml_slope,
               vancomycin_segments_n, amikacin_segments_n,
               gentamicin_segments_n, tobramycin_segments_n
        FROM features WHERE stay_id = 100 ORDER BY prediction_time
    """).fetchall()
    assert isclose(renal[0][0], 0.2) and isclose(renal[0][1], 30)
    assert renal[0][2:] == (1, 0, 0, 0)
    assert renal[1] == (None, None, 0, 0, 0, 1)
    print("Synthetic expected-value, leakage, boundary and overlapping-stay checks passed.")

In [ ]:
if SHOW_STAY_TRACE:
    stay = TRACE_STAY_ID
    if stay is None:
        stay = con.execute("""
            SELECT stay_id FROM features ORDER BY (heart_rate_n > 0) DESC, stay_id LIMIT 1
        """).fetchone()[0]
    result = con.execute("""
        SELECT * FROM features WHERE stay_id = ?
        ORDER BY (heart_rate_n > 0) DESC, prediction_time LIMIT 1
    """, [stay])
    fields = [field[0] for field in result.description]
    row = result.fetchone()
    if row is None:
        raise ValueError("TRACE_STAY_ID has no feature rows in this cohort.")
    feature_row = dict(zip(fields, row))
    print(feature_row)
    result = con.execute("""
        SELECT feature, itemid, charttime, storetime, available_time, hour_index, raw_value, valueuom,
               feature_value IS NOT NULL AND coalesce(available_time <= ?, false)
                   AND charttime < ? AS eligible_for_window
        FROM hourly_observations
        WHERE stay_id = ? AND charttime >= ? AND charttime <= ?
        ORDER BY charttime, feature, itemid, storetime, raw_value LIMIT 100
    """, [feature_row["prediction_time"], feature_row["prediction_time"], stay,
          feature_row["window_start"], feature_row["prediction_time"]])
    print([field[0] for field in result.description])
    for row in result.fetchall():
        print(row)
else:
    print("Stay trace is off. Enable SHOW_STAY_TRACE in the local copy to inspect records.")

## Hand off features for labels

Join Tanusha's ICU event timestamps to the feature table by stay_id. Keep (stay_id, prediction_time) unique. For each future 6-, 12-, or 24-hour horizon, compare the onset with (prediction_time, prediction_time + horizon]. Decide eligibility at the cutoff and how to handle incomplete follow-up. First incident AKI requires a different risk set from progression from Stage 1. Remove windows at or after the relevant onset and audit near-onset creatinine and urine overlap against the label timestamps. subject_id supports patient-level splitting. outtime and other timestamps are join or audit metadata. If the label data has multiple events per stay, reduce it to the intended event per horizon or per cutoff before merging, so the feature key remains unique. Imputation and scaling belong downstream and must be fit on training data only.